In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, cohen_kappa_score
import lightgbm as lgb
import itertools

ROOT=Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
with open(ROOT/'config.yaml', 'r') as file:
    config=yaml.safe_load(file)

df=pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_usable=df[df.usable].copy()
df_usable=df_usable[df_usable['label'].notna()]

meta_cols=['participant', 'epoch_number', 'label', 'usable', 'reason', 'label']
feature_cols=[c for c in df_usable.columns if c not in meta_cols]

X=np.ascontiguousarray(df_usable[feature_cols].values, dtype=np.float32)
y=df_usable['label'].values
groups=df_usable['participant'].values
classes=np.array(['W', 'N1', 'N2', 'N3', 'R'])

sgkf_outer=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=config.get('seed', 42))
outer_folds=list(sgkf_outer.split(X, y, groups))

In [21]:
import warnings

param_grid={'num_leaves':[31,63], 'learning_rate':[0.03,0.1], 'min_child_samples':[20,50]}
keys, values=zip(*param_grid.items())
grid_combinations=[dict(zip(keys, v)) for v in itertools.product(*values)]

outer_macro_f1s, outer_bal_accs, outer_kappas=[], [], []
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", message=".*F-score is ill-defined.*")
    for index,(train_idx_outer,val_idx_outer) in enumerate(outer_folds):
        X_tr_out, y_tr_out, g_tr_out=X[train_idx_outer], y[train_idx_outer], groups[train_idx_outer]
        X_val_out, y_val_out=X[val_idx_outer], y[val_idx_outer]

        sgkf_inner=StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=config.get('seed', 42))
        inner_folds=list(sgkf_inner.split(X_tr_out, y_tr_out, g_tr_out))

        best_inner_score, best_params=-1.0, None
        for params in grid_combinations:
            inner_f1s=[]
            for train_idx_in, val_idx_in in inner_folds:
                X_tr_in, y_tr_in=X_tr_out[train_idx_in], y_tr_out[train_idx_in]
                X_val_in, y_val_in=X_tr_out[val_idx_in], y_tr_out[val_idx_in]

                model_in=lgb.LGBMClassifier(**params,class_weight='balanced',n_estimators=100,random_state=config.get('seed', 42),n_jobs=-1,verbose=-1)
                model_in.fit(X_tr_in, y_tr_in)
                preds_in=model_in.predict(X_val_in)
                inner_f1s.append(f1_score(y_val_in, preds_in, average='macro', labels=classes,zero_division=0))

            mean_inner_f1=np.mean(inner_f1s)
            if mean_inner_f1 > best_inner_score:
                best_inner_score=mean_inner_f1
                best_params=params

        model_out=lgb.LGBMClassifier(**best_params,class_weight='balanced',n_estimators=100,random_state=config.get('seed', 42),n_jobs=-1,verbose=-1)
        model_out.fit(X_tr_out, y_tr_out)
        preds_out=model_out.predict(X_val_out)

        outer_macro_f1s.append(f1_score(y_val_out, preds_out, average='macro', labels=classes, zero_division=0))
        outer_bal_accs.append(balanced_accuracy_score(y_val_out, preds_out))
        outer_kappas.append(cohen_kappa_score(y_val_out, preds_out, labels=classes))

        print(f"Outer Fold {index+1}/5 Selected Params: {best_params} | Inner F1: {best_inner_score:.4f}")

print("\nLIGHTGBM(NESTED TUNED)")
print(f"\tMacro-F1: {np.mean(outer_macro_f1s):.4f} ± {np.std(outer_macro_f1s):.4f}")
print(f"\tBalanced Accuracy: {np.mean(outer_bal_accs):.4f}")
print(f"\tCohen's Kappa: {np.mean(outer_kappas):.4f}")

Outer Fold 1/5 Selected Params: {'num_leaves': 31, 'learning_rate': 0.1, 'min_child_samples': 20} | Inner F1: 0.3579
Outer Fold 2/5 Selected Params: {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 50} | Inner F1: 0.3362
Outer Fold 3/5 Selected Params: {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 20} | Inner F1: 0.3390
Outer Fold 4/5 Selected Params: {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 50} | Inner F1: 0.3416
Outer Fold 5/5 Selected Params: {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 20} | Inner F1: 0.3434

LIGHTGBM(NESTED TUNED)
	Macro-F1: 0.3524 ± 0.0072
	Balanced Accuracy: 0.3733
	Cohen's Kappa: 0.2503


In [25]:
import warnings
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, balanced_accuracy_score, cohen_kappa_score
import lightgbm as lgb

ROOT=Path.cwd().resolve().parent
with open(ROOT/'config.yaml', 'r') as file:
    config=yaml.safe_load(file)

df=pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_usable=df[df.usable].copy()
df_usable=df_usable[df_usable['label'].notna()].copy()

groups=df_usable['participant'].values
classes=np.array(['W', 'N1', 'N2', 'N3', 'R'])
df_folds=pd.read_csv(ROOT/'results'/'folds.csv')

folds=[]
for fold_idx in sorted(df_folds['fold'].unique()):
    val_parts=df_folds[df_folds['fold'] == fold_idx]['participant'].values
    val_mask=df_usable['participant'].isin(val_parts)
    train_idx=np.where(~val_mask)[0]
    val_idx=np.where(val_mask)[0]
    folds.append((train_idx, val_idx))

all_feature_cols=[c for c in df_usable.columns if c not in ['participant', 'epoch_number', 'label', 'usable', 'reason', 'label']]
hr_base = ['HR', 'ibi', 'sdnn', 'rmssd', 'pnn50', 'beat_count', 'dominant_freq']
acc_base = ['mag', 'jerk', 'sd_x', 'sd_y', 'sd_z', 'share_away_1g', 'angle_change']
eda_temp_base = ['eda', 'temp']
hr_cols = [c for c in all_feature_cols if any(k in c for k in hr_base)]
acc_cols = [c for c in all_feature_cols if any(k in c for k in acc_base)]
eda_temp_cols = [c for c in all_feature_cols if any(k in c for k in eda_temp_base)]

ablation_configs={'A1(HR/HRV only)': hr_cols,'A2(Movement only)': acc_cols,'A3(EDA+TEMP only)': eda_temp_cols,'A4(All Combined)': all_feature_cols}

results={}
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    
    for config_name, feature_subset in ablation_configs.items():
        X_sub=np.ascontiguousarray(df_usable[feature_subset].values, dtype=np.float32)
        y=df_usable['label'].values
        
        fold_macro_f1s=[]
        fold_bal_accs=[]
        fold_kappas=[]
        
        for train_idx, val_idx in folds:
            X_tr, y_tr=X_sub[train_idx], y[train_idx]
            X_val, y_val=X_sub[val_idx], y[val_idx]
            
            model=lgb.LGBMClassifier(n_estimators=100,learning_rate=0.05,num_leaves=31,class_weight='balanced',random_state=config.get('seed', 42),n_jobs=-1,verbose=-1)
            model.fit(X_tr, y_tr)
            preds=model.predict(X_val)
            
            fold_macro_f1s.append(f1_score(y_val, preds, average='macro', labels=classes, zero_division=0))
            fold_bal_accs.append(balanced_accuracy_score(y_val, preds))
            fold_kappas.append(cohen_kappa_score(y_val, preds, labels=classes))
            
        results[config_name]={'Macro_F1_Mean': np.mean(fold_macro_f1s),'Macro_F1_SD': np.std(fold_macro_f1s),'Bal_Acc_Mean': np.mean(fold_bal_accs),'Kappa_Mean': np.mean(fold_kappas),'Fold_F1s': fold_macro_f1s}

summary_rows=[]
for name, res in results.items():
    summary_rows.append({'Configuration': name,'Macro-F1(Mean ± SD)': f"{res['Macro_F1_Mean']:.4f} ± {res['Macro_F1_SD']:.4f}",'Balanced Accuracy': f"{res['Bal_Acc_Mean']:.4f}",'Cohen\'s Kappa': f"{res['Kappa_Mean']:.4f}"})

df_summary=pd.DataFrame(summary_rows)
print("SENSOR ABLATION RESULTS(DREAMT)")
print(df_summary)

best_single_name=max(['A1(HR/HRV only)', 'A2(Movement only)', 'A3(EDA+TEMP only)'], key=lambda k: results[k]['Macro_F1_Mean'])
best_single_f1=results[best_single_name]['Macro_F1_Mean']
combined_f1=results['A4(All Combined)']['Macro_F1_Mean']
combined_sd=results['A4(All Combined)']['Macro_F1_SD']

margin=combined_f1-best_single_f1
if margin > combined_sd:
    print(f"Outcome: Multimodal combination helps.")
    print(f"Reason: Combined A4({combined_f1:.4f}) outperforms best single-sensor {best_single_name}({best_single_f1:.4f}) by {margin:.4f}, which exceeds the fold-to-fold SD threshold({combined_sd:.4f}).")
else:
    print(f"Outcome: Multimodal combination provides no unambiguous gain.")
    print(f"Reason: Gain over best single sensor({margin:.4f}) does not exceed the fold-to-fold spread({combined_sd:.4f}).")

SENSOR ABLATION RESULTS(DREAMT)
       Configuration Macro-F1(Mean ± SD) Balanced Accuracy Cohen's Kappa
0    A1(HR/HRV only)     0.2694 ± 0.0089            0.3205        0.1392
1  A2(Movement only)     0.2964 ± 0.0084            0.3308        0.1765
2  A3(EDA+TEMP only)     0.2644 ± 0.0151            0.2842        0.1296
3   A4(All Combined)     0.3500 ± 0.0106            0.3773        0.2471
Outcome: Multimodal combination helps.
Reason: Combined A4(0.3500) outperforms best single-sensor A2(Movement only)(0.2964) by 0.0536, which exceeds the fold-to-fold SD threshold(0.0106).


In [23]:
import os
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
with open(ROOT/'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

tables_dir = ROOT/'results'/'tables'
figures_dir = ROOT/'results'/'figures'
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_folds = pd.read_csv(ROOT/'results'/'folds.csv')

stage_map = {'W': 'W', 'R': 'R', 'Wake': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'REM': 'R'}
df_usable = df[df.usable].copy()
df_usable['target'] = df_usable['label'].map(stage_map)
df_usable = df_usable[df_usable['target'].notna()].copy()

df_usable = df_usable.merge(df_folds, on='participant', how='left')
print("LEAKAGE AUDIT")
leakage_detected = False

for fold_idx in sorted(df_usable['fold'].unique()):
    train_parts = set(df_usable[df_usable['fold'] != fold_idx]['participant'])
    test_parts = set(df_usable[df_usable['fold'] == fold_idx]['participant'])
    overlap = train_parts.intersection(test_parts)
    
    print(f"\tFold {fold_idx}: Train Participants = {len(train_parts)}\n\t\tTest Participants = {len(test_parts)}\n\t\tOverlap = {len(overlap)}")
    if len(overlap)>0:
        leakage_detected = True

if not leakage_detected:
    print("Success: 0 overlaps found across all 5 folds.\n")
else:
    raise ValueError("LEAKAGE ERROR: Participant overlap detected between train and test folds.")

meta_cols = ['participant', 'epoch_number', 'label', 'usable', 'reason', 'target', 'fold']
feature_cols = [c for c in df_usable.columns if c not in meta_cols]

oof_df = df_usable[['participant', 'epoch_number', 'target', 'fold']].copy()
oof_df.to_parquet(tables_dir/'dreamt_oof_manifest.parquet', index=False)
print(f"OOF Manifest saved to: {tables_dir/'dreamt_oof_manifest.parquet'}(Total Epochs: {len(oof_df)})")

LEAKAGE AUDIT
	Fold 0: Train Participants = 80
		Test Participants = 20
		Overlap = 0
	Fold 1: Train Participants = 80
		Test Participants = 20
		Overlap = 0
	Fold 2: Train Participants = 80
		Test Participants = 20
		Overlap = 0
	Fold 3: Train Participants = 79
		Test Participants = 21
		Overlap = 0
	Fold 4: Train Participants = 81
		Test Participants = 19
		Overlap = 0
Success: 0 overlaps found across all 5 folds.

OOF Manifest saved to: D:\Adobe\projects\research proj\results\tables\dreamt_oof_manifest.parquet(Total Epochs: 79074)


In [ ]:
import json
import joblib
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

ROOT = Path.cwd().resolve().parent
with open(ROOT/'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

frozen_dir = ROOT/'results'/'frozen_model'
frozen_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')

stage_map = {'W': 'W', 'R': 'R', 'Wake': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'REM': 'R'}
df_usable = df[df.usable].copy()
df_usable['target'] = df_usable['label'].map(stage_map)
df_usable = df_usable[df_usable['target'].notna()].copy()

meta_cols = ['participant', 'epoch_number', 'label', 'usable', 'reason', 'target', 'fold']
feature_cols = [c for c in df_usable.columns if c not in meta_cols]

X_full = np.ascontiguousarray(df_usable[feature_cols].values, dtype=np.float32)

le = LabelEncoder()
classes_str = ['W', 'N1', 'N2', 'N3', 'R']
le.fit(classes_str)
y_full = le.transform(df_usable['target'].values)
best_params = {'num_leaves': 31,'learning_rate': 0.05,'min_child_samples': 20,'n_estimators': 100,'class_weight': 'balanced','objective': 'multiclass','num_class': 5,'random_state': config.get('seed', 42),'n_jobs': -1,'verbose': -1}

print("Training final frozen model on DREAMT data.")
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_full, y_full)

model_path = frozen_dir/'final_lightgbm_model.pkl'
joblib.dump(final_model, model_path)
meta_manifest = {'model_type': 'LightGBMClassifier','target_classes': list(le.classes_),'feature_names': feature_cols,'num_features': len(feature_cols),'hyperparameters': best_params,'training_dataset': 'DREAMT(All 100 participants, Usable Epochs)','total_training_epochs': len(df_usable)}

with open(frozen_dir/'model_metadata.json', 'w') as f:
    json.dump(meta_manifest, f, indent=4)

print(f"Frozen model saved to {model_path}")
print(f"Model metadata exported to {frozen_dir/'model_metadata.json'}\n")

Training final frozen model on DREAMT data.
Frozen model saved to D:\Adobe\projects\research proj\results\frozen_model\final_lightgbm_model.pkl
Model metadata exported to D:\Adobe\projects\research proj\results\frozen_model\model_metadata.json

